# Newton's Form and the Divided-Difference Table

The polynomial through a given set of points is unique, but there is more than one way to write it down. The Runge notebook used the Lagrange form. This one builds the same polynomial in Newton's form, where the coefficients come out of a divided-difference table and adding a data point costs one extra term.

In [ ]:
# --- Colab / Jupyter setup ------------------------------------------------
# Enable interactive ipywidgets sliders. try/except so the notebook also runs
# in plain Jupyter (outside Colab) without error.
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except Exception:
    pass

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider, Dropdown, Checkbox

%matplotlib inline
plt.rcParams["figure.figsize"] = (9, 5)

## Divided differences

$$ p_n(x) = \sum_{k=0}^{n} f[x_0,\dots,x_k]\prod_{j=0}^{k-1}(x - x_j), \qquad f[x_i,\dots,x_{i+j}] = \frac{f[x_{i+1},\dots,x_{i+j}] - f[x_i,\dots,x_{i+j-1}]}{x_{i+j}-x_i}. $$

The coefficients $f[x_0,\dots,x_k]$ are the top row of the triangular table. Add a node and the table gains one anti-diagonal while every coefficient already computed stays where it was. That is what makes Newton's form convenient for data that arrive a point at a time.

In [ ]:
# ---------------------------------------------------------------------------
# The divided-difference table
# ---------------------------------------------------------------------------
# Newton's form writes the interpolating polynomial as
#
#   p(x) = c0 + c1 (x-x0) + c2 (x-x0)(x-x1) + ... + cn (x-x0)...(x-x_{n-1})
#
# where the coefficients c_k are "divided differences" f[x0,...,xk]. They are
# built from a triangular table, each entry a difference of two neighbors in
# the previous column, divided by the spread of the x's involved.

def divided_differences(x, y):
    """Build the divided-difference table for data (x[i], y[i]).

    Parameters
    ----------
    x : array (n,)  distinct nodes
    y : array (n,)  values, y[i] = f(x[i])

    Returns
    -------
    coeffs : array (n,)   Newton coefficients c0..c_{n-1} (the top row of the table)
    table  : array (n,n)  full lower-triangular table (zeros above the diagonal used)

    The recurrence is  table[i, j] = ( table[i+1, j-1] - table[i, j-1] )
                                   / ( x[i+j] - x[i] )
    Column 0 is just the y-values; each later column is one order higher.
    """
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    n = len(x)
    table = np.zeros((n, n))
    table[:, 0] = y                          # 0th divided differences are the y's
    for j in range(1, n):                    # column = order of the difference
        for i in range(n - j):               # row
            table[i, j] = (table[i + 1, j - 1] - table[i, j - 1]) / (x[i + j] - x[i])
    coeffs = table[0, :].copy()              # Newton coefficients live on the top row
    return coeffs, table


def newton_eval(x_nodes, coeffs, x):
    """Evaluate the Newton-form polynomial with the given coefficients at x.

    Uses a nested (Horner-like) scheme, which costs O(n) and is numerically
    steadier than expanding the products explicitly.

        p = c_{n-1}
        p = p*(x - x_{n-2}) + c_{n-2}
        ...
        p = p*(x - x_0) + c_0
    """
    x = np.asarray(x, dtype=float)
    n = len(coeffs)
    result = np.full_like(x, coeffs[-1])     # start from the highest coefficient
    for k in range(n - 2, -1, -1):           # fold in the rest, high to low
        result = result * (x - x_nodes[k]) + coeffs[k]
    return result

In [ ]:
# ---------------------------------------------------------------------------
# A readable printout of the triangular table
# ---------------------------------------------------------------------------
def print_dd_table(x, table):
    """Pretty-print the divided-difference table as a triangle.

    Row i, column j holds f[x_i, ..., x_{i+j}]. The Newton coefficients are the
    top row (j-th coefficient = table[0, j]).
    """
    n = len(x)
    print("  x_i   |  f[.] (order increases left -> right)")
    print("-" * 60)
    for i in range(n):
        row = f"{x[i]:6.3f} |"
        for j in range(n - i):
            row += f" {table[i, j]:10.4f}"
        print(row)

In [ ]:
# ---------------------------------------------------------------------------
# Incremental construction
# ---------------------------------------------------------------------------
# Append one more node to the end of the list and only one new coefficient
# appears; the earlier ones are unchanged, since c_k depends only on x_0..x_k.
# (Lagrange, by contrast, must rebuild every basis polynomial from scratch.)
# The reuse needs the old nodes to survive in the same order. Rebuilding an
# equally spaced set at each size moves every node, and then every coefficient
# changes. The cell after this one shows both cases side by side.

def f_demo(x):
    """Sample function, smooth and oscillatory."""
    return np.sin(2 * x) + 0.5 * x

def show_newton(n_nodes=5, show_table=True, a=0.0, b=5.0):
    """Sample f_demo at n_nodes equally spaced points, build the Newton
    interpolant, plot it against f, and optionally print the table + coeffs."""
    x = np.linspace(a, b, n_nodes)           # nodes
    y = f_demo(x)                            # data values
    coeffs, table = divided_differences(x, y)

    xx = np.linspace(a, b, 400)
    p = newton_eval(x, coeffs, xx)

    plt.figure()
    plt.plot(xx, f_demo(xx), "k-", lw=2, label="f(x)")
    plt.plot(xx, p, "r--", lw=2, label=f"Newton interpolant (n={n_nodes})")
    plt.plot(x, y, "bo", ms=7, label="data")
    plt.legend(); plt.xlabel("x"); plt.title("Newton-form interpolation")
    plt.show()

    if show_table:
        print_dd_table(x, table)
        print("\nNewton coefficients c0..c{}:".format(n_nodes - 1))
        print(np.array2string(coeffs, precision=4))
        print("\nNote. These nodes are rebuilt as an equally spaced set at every "
              "size, so\nevery coefficient moves when n changes. See the next cell "
              "for the appended case.")

show_newton(5)

In [ ]:
# ---------------------------------------------------------------------------
# Interactive: slide the number of nodes; toggle the table
# ---------------------------------------------------------------------------
interact(
    show_newton,
    n_nodes=IntSlider(min=2, max=12, step=1, value=5, description="# nodes"),
    show_table=Checkbox(value=True, description="show table"),
    a=(-1.0, 2.0, 0.5), b=(3.0, 8.0, 0.5),
);

In [ ]:
# ---------------------------------------------------------------------------
# Appended nodes versus a rebuilt equally spaced set
# ---------------------------------------------------------------------------
def coeffs_for(nodes):
    """Newton coefficients for f_demo sampled at these nodes, in this order."""
    nodes = np.asarray(nodes, dtype=float)
    c, _ = divided_differences(nodes, f_demo(nodes))
    return c

appended = [0.0, 2.5, 5.0, 1.25, 3.75, 0.625, 1.875, 3.125, 4.375]
print("each new node appended to the end of the list")
for m in [3, 5, 9]:
    print(f"  m = {m}  {np.array2string(coeffs_for(appended[:m]), precision=4)}")

print("\nan equally spaced set rebuilt from scratch at each size")
for m in [3, 5, 9]:
    print(f"  m = {m}  {np.array2string(coeffs_for(np.linspace(0, 5, m)), precision=4)}")

## Divided differences and derivatives

A first divided difference is a difference quotient, so the Mean Value Theorem gives $f[x_0,x_1] = f'(\eta)$ for some $\eta$ between the nodes. Higher orders behave the same way. Let all $k+1$ nodes run together at $x_0$ and $f[x_0,\dots,x_k]$ tends to $f^{(k)}(x_0)/k!$, the $k$th Taylor coefficient. The cell below checks that for $k = 1, 2, 3$.

In [ ]:
x0  = 1.0
fp  = 2 * np.cos(2 * x0) + 0.5          # first derivative of f_demo at x0
fpp = -4 * np.sin(2 * x0)               # second derivative
f3  = -8 * np.cos(2 * x0)               # third derivative

print(f"{'h':>9} {'order 1':>14} {'order 2':>14} {'order 3':>14}")
for h in [1e-1, 1e-2, 1e-3, 1e-4]:
    row = f"{h:9.0e}"
    for k in [1, 2, 3]:
        xs = x0 + h * np.arange(k + 1)      # k+1 nodes collapsing onto x0
        ck, _ = divided_differences(xs, f_demo(xs))
        row += f" {ck[k]:14.8f}"
    print(row)

print("\nTaylor coefficients of f at x0")
print(f"  first  derivative      = {fp:.8f}")
print(f"  second derivative / 2  = {fpp / 2:.8f}")
print(f"  third  derivative / 6  = {f3 / 6:.8f}")

## Coincident nodes and roundoff

The recurrence divides by $x_{i+j}-x_i$. Bring two nodes close together and you are dividing a difference of nearly equal numbers by a nearly zero spacing, and both operations cost digits. In the sweep below the error follows the truncation term while $h$ is large, then turns and follows $\varepsilon/h$ once cancellation takes over ($\varepsilon$ is the unit roundoff). The interpolating polynomial itself is perfectly well behaved for these data, so the damage belongs to the algorithm and not to the mathematics. The same trade-off returns in numerical differentiation, where it fixes an optimal step size.

In [ ]:
eps = np.finfo(float).eps

print(f"exact f'(x0) = {fp:.12f}   unit roundoff = {eps:.2e}\n")
print(f"{'h':>9} {'f[x0,x0+h]':>18} {'error':>12} {'eps/h':>12}")
for h in [1e-1, 1e-4, 1e-8, 1e-11, 1e-13, 1e-15]:
    xs = np.array([x0, x0 + h])
    c, _ = divided_differences(xs, f_demo(xs))
    print(f"{h:9.0e} {c[1]:18.10f} {abs(c[1] - fp):12.2e} {eps / h:12.2e}")

## Summary

Newton and Lagrange are two representations of the same unique polynomial. Building the divided-difference table costs $\mathcal{O}(n^2)$, and evaluating by nested multiplication costs $\mathcal{O}(n)$. Appending a node at the end of the list adds exactly one coefficient and reuses every earlier one, though reordering or replacing nodes does not. As nodes coalesce the divided differences approach Taylor coefficients, but in floating point the recurrence gives out well before they get there.

## Things to try

- Raise the node count one step at a time. The slider rebuilds an equally spaced set, so every coefficient moves; the appended-node cell above is the case where the earlier entries stay put.
- Edit the node array to put two nodes almost on top of each other, then watch the high-order columns of the table.
- Push the node count to 12 and Runge's phenomenon shows up. It does not care which representation of the polynomial you chose.